In [2]:
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

base_dir = r'C:\Users\Admin\Downloads\Compressed\FaceForensics++\data_flat'

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
)

train_generator = datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

validation_generator = datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)


Found 8453 images belonging to 2 classes.
Found 2112 images belonging to 2 classes.


In [4]:
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(optimizer=Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])

model.fit(train_generator, validation_data=validation_generator, epochs=15)

model.save('mobilenetv2.h5')


9406464/9406464 [==============================] - 5s 1us/step
Epoch 1/15
265/265 [==============================] - 702s 3s/step - loss: 0.6821 - accuracy: 0.6039 - val_loss: 0.7716 - val_accuracy: 0.4882
Epoch 2/15
265/265 [==============================] - 737s 3s/step - loss: 0.5896 - accuracy: 0.6856 - val_loss: 0.8136 - val_accuracy: 0.4844
Epoch 3/15
265/265 [==============================] - 710s 3s/step - loss: 0.5584 - accuracy: 0.7104 - val_loss: 0.8882 - val_accuracy: 0.4602
Epoch 4/15
265/265 [==============================] - 709s 3s/step - loss: 0.5307 - accuracy: 0.7307 - val_loss: 0.9042 - val_accuracy: 0.4574
Epoch 5/15
265/265 [==============================] - 736s 3s/step - loss: 0.5221 - accuracy: 0.7320 - val_loss: 0.9234 - val_accuracy: 0.4669
Epoch 6/15
265/265 [==============================] - 740s 3s/step - loss: 0.5058 - accuracy: 0.7459 - val_loss: 0.9639 - val_accuracy: 0.4759
Epoch 7/15
265/265 [==============================] - 731s 3s/step - loss: 0.48

C:\Users\Admin\anaconda3\envs\deepfake_detection\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [6]:
# Evaluate the model on the validation data
val_loss, val_accuracy = model.evaluate(validation_generator)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

from sklearn.metrics import confusion_matrix, classification_report

# Reset the validation generator to ensure predictions start from the beginning
validation_generator.reset()

# Get predictions on the validation set
preds = model.predict(validation_generator)
predicted_classes = (preds > 0.5).astype(int).ravel()

# Get true class labels
true_classes = validation_generator.classes
class_labels = list(validation_generator.class_indices.keys())

# Generate and print the confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)
print("Confusion Matrix:")
print(cm)

# Generate and print the classification report
report = classification_report(true_classes, predicted_classes, target_names=class_labels)
print("Classification Report:")
print(report)



66/66 [==============================] - 91s 1s/step - loss: 1.1327 - accuracy: 0.4645
Validation Loss: 1.1327
Validation Accuracy: 0.4645
66/66 [==============================] - 96s 1s/step
Confusion Matrix:
[[732 234]
 [860 286]]
Classification Report:
              precision    recall  f1-score   support

        fake       0.46      0.76      0.57       966
        real       0.55      0.25      0.34      1146

    accuracy                           0.48      2112
   macro avg       0.50      0.50      0.46      2112
weighted avg       0.51      0.48      0.45      2112

